# Household Sound Classifier — Training Notebook

Train a custom CNN on mel spectrograms to classify household sounds.

**Dataset:** ESC-50 (2,000 clips, 50 classes) + custom recorded clips

**Model:** 4-block CNN (~150K params)

**Authors:** Joshua Kirby & Alan Nur (with Claude Opus 4.6 LLM assistance)  
**Course:** TECHIN 513A — Managing Data And Signal Processing

## 1. Setup & Dependencies

In [ ]:
# Install dependencies (run this cell first on Colab)
# !pip install torch torchaudio numpy pandas matplotlib scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import json
import random

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Download ESC-50 Dataset

In [ ]:
# Download ESC-50 (only run once)
!git clone https://github.com/karolpiczak/ESC-50.git data/ESC-50 2>/dev/null || echo "Already downloaded."
!ls data/ESC-50/audio/ | head -5
!wc -l data/ESC-50/meta/esc50.csv

In [ ]:
# Load metadata
meta = pd.read_csv("data/ESC-50/meta/esc50.csv")
print(f"Total clips: {len(meta)}")
print(f"Classes: {meta['category'].nunique()}")
print()
print("All categories:")
for i, cat in enumerate(sorted(meta['category'].unique())):
    print(f"  {i:2d}. {cat}")

## 3. Prepare Dataset (ESC-50 + Custom Clips)

In [ ]:
# ── Configuration ──
SAMPLE_RATE = 44100
DURATION = 5          # seconds (ESC-50 standard)
N_SAMPLES = SAMPLE_RATE * DURATION
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
F_MAX = 22050
TOP_DB = 80

# Mel spectrogram transform
mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_max=F_MAX,
)
amp_to_db = T.AmplitudeToDB(top_db=TOP_DB)

print(f"Expected mel spectrogram shape: (1, {N_MELS}, {N_SAMPLES // HOP_LENGTH + 1})")

In [ ]:
class SoundDataset(Dataset):
    """Dataset that loads ESC-50 clips + optional custom clips as mel spectrograms."""

    def __init__(self, file_list, label_list, augment=False):
        self.files = file_list
        self.labels = label_list
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def _load_audio(self, path):
        """Load audio, convert to mono, resample to target rate, pad/trim to fixed length."""
        waveform, sr = torchaudio.load(path)
        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        # Resample if needed
        if sr != SAMPLE_RATE:
            resampler = T.Resample(sr, SAMPLE_RATE)
            waveform = resampler(waveform)
        # Pad or trim to exactly N_SAMPLES
        if waveform.shape[1] < N_SAMPLES:
            pad = N_SAMPLES - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad))
        else:
            waveform = waveform[:, :N_SAMPLES]
        return waveform  # (1, N_SAMPLES)

    def _augment(self, waveform):
        """Apply random augmentations to the waveform."""
        # Time shift
        if random.random() < 0.5:
            shift = random.randint(-SAMPLE_RATE, SAMPLE_RATE)  # up to 1 sec
            waveform = torch.roll(waveform, shift, dims=-1)
        # Add noise
        if random.random() < 0.5:
            noise = torch.randn_like(waveform) * 0.005
            waveform = waveform + noise
        return waveform

    def __getitem__(self, idx):
        waveform = self._load_audio(self.files[idx])
        if self.augment:
            waveform = self._augment(waveform)
        mel = mel_transform(waveform)
        mel_db = amp_to_db(mel)  # (1, N_MELS, T)
        return mel_db, self.labels[idx]

In [ ]:
def build_file_lists():
    """
    Build lists of (filepath, label_index) for ESC-50 + custom clips.
    Returns: files, labels, label_names, fold_assignments
    """
    files, labels, folds = [], [], []

    # ── ESC-50 ──
    meta = pd.read_csv("data/ESC-50/meta/esc50.csv")
    esc_categories = sorted(meta['category'].unique())
    label_names = list(esc_categories)  # start with ESC-50 classes
    cat_to_idx = {c: i for i, c in enumerate(label_names)}

    for _, row in meta.iterrows():
        path = os.path.join("data", "ESC-50", "audio", row['filename'])
        if os.path.exists(path):
            files.append(path)
            labels.append(cat_to_idx[row['category']])
            folds.append(row['fold'])

    print(f"ESC-50: {len(files)} clips, {len(label_names)} classes")

    # ── Custom clips ──
    custom_dir = "data/custom"
    if os.path.exists(custom_dir):
        for class_name in sorted(os.listdir(custom_dir)):
            class_path = os.path.join(custom_dir, class_name)
            if not os.path.isdir(class_path):
                continue
            # Add new class if not already in ESC-50
            if class_name not in cat_to_idx:
                cat_to_idx[class_name] = len(label_names)
                label_names.append(class_name)
            idx = cat_to_idx[class_name]
            wavs = [f for f in os.listdir(class_path) if f.endswith('.wav')]
            for wav in wavs:
                files.append(os.path.join(class_path, wav))
                labels.append(idx)
                # Assign custom clips to training folds (1-4)
                folds.append(random.randint(1, 4))
            print(f"Custom class '{class_name}': {len(wavs)} clips (label {idx})")

    print(f"\nTotal: {len(files)} clips, {len(label_names)} classes")
    return files, labels, label_names, folds


files, labels, label_names, folds = build_file_lists()
NUM_CLASSES = len(label_names)
print(f"\nNum classes: {NUM_CLASSES}")

In [ ]:
# ── Train/Val/Test split using ESC-50 folds ──
# Fold 5 = test, Fold 4 = validation, Folds 1-3 = training
TEST_FOLD = 5
VAL_FOLD = 4

train_files, train_labels = [], []
val_files, val_labels = [], []
test_files, test_labels = [], []

for f, l, fold in zip(files, labels, folds):
    if fold == TEST_FOLD:
        test_files.append(f)
        test_labels.append(l)
    elif fold == VAL_FOLD:
        val_files.append(f)
        val_labels.append(l)
    else:
        train_files.append(f)
        train_labels.append(l)

print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

train_dataset = SoundDataset(train_files, train_labels, augment=True)
val_dataset = SoundDataset(val_files, val_labels, augment=False)
test_dataset = SoundDataset(test_files, test_labels, augment=False)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# ── Visualize a sample mel spectrogram ──
sample_mel, sample_label = train_dataset[0]
print(f"Mel shape: {sample_mel.shape}")
print(f"Label: {sample_label} ({label_names[sample_label]})")

fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(sample_mel.squeeze().numpy(), aspect='auto', origin='lower', cmap='magma')
ax.set_xlabel('Time Frame')
ax.set_ylabel('Mel Bin')
ax.set_title(f'Mel Spectrogram: {label_names[sample_label]}')
plt.tight_layout()
plt.show()

## 4. Define CNN Model

In [ ]:
class HouseholdSoundCNN(nn.Module):
    """4-block CNN for classifying mel spectrograms."""

    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Global average pooling
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = HouseholdSoundCNN(NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(model)

## 5. Training Loop

In [ ]:
# ── Training configuration ──
EPOCHS = 80
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  factor=0.5, patience=5)

# ── SpecAugment (applied to mel spectrograms during training) ──
freq_mask = T.FrequencyMasking(freq_mask_param=15)
time_mask = T.TimeMasking(time_mask_param=35)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for mels, labels in loader:
        mels, labels = mels.to(DEVICE), labels.to(DEVICE)
        # SpecAugment
        mels = freq_mask(mels)
        mels = time_mask(mels)
        optimizer.zero_grad()
        outputs = model(mels)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * mels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += mels.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for mels, labels in loader:
            mels, labels = mels.to(DEVICE), labels.to(DEVICE)
            outputs = model(mels)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * mels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += mels.size(0)
    return total_loss / total, correct / total

In [ ]:
# ── Train! ──
best_val_acc = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pt")

    if (epoch + 1) % 5 == 0 or epoch == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train: {train_loss:.4f} / {train_acc:.1%} | "
              f"Val: {val_loss:.4f} / {val_acc:.1%} | "
              f"LR: {lr:.1e}")

print(f"\nBest validation accuracy: {best_val_acc:.1%}")

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 7. Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"Test accuracy: {test_acc:.1%}")

In [ ]:
# ── Confusion matrix ──
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for mels, labels in test_loader:
        mels = mels.to(DEVICE)
        preds = model(mels).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

# Only show classes that appear in the test set
present = sorted(set(all_labels))
present_names = [label_names[i] for i in present]

cm = confusion_matrix(all_labels, all_preds, labels=present)
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(present_names)))
ax.set_yticks(range(len(present_names)))
ax.set_xticklabels(present_names, rotation=90, fontsize=7)
ax.set_yticklabels(present_names, fontsize=7)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix (Test Accuracy: {test_acc:.1%})')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print("\nPer-class report:")
print(classification_report(all_labels, all_preds,
                            labels=present, target_names=present_names))

## 8. Export Model & Labels

In [ ]:
# Save final model and label mapping for the classifier script
os.makedirs("models", exist_ok=True)

# Model weights
torch.save(model.state_dict(), "models/model.pt")
print("Saved: models/model.pt")

# Label mapping
label_map = {"num_classes": NUM_CLASSES, "labels": label_names}
with open("models/labels.json", "w") as f:
    json.dump(label_map, f, indent=2)
print("Saved: models/labels.json")

# Training config (so classifier uses matching mel params)
config = {
    "sample_rate": SAMPLE_RATE,
    "duration": DURATION,
    "n_fft": N_FFT,
    "hop_length": HOP_LENGTH,
    "n_mels": N_MELS,
    "f_max": F_MAX,
    "top_db": TOP_DB,
}
with open("models/config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Saved: models/config.json")

print(f"\nDone! Test accuracy: {test_acc:.1%}")
print("Download models/ folder and place it in your menial_ai project root.")

## 9. (Optional) Download to Local Machine

If running in Google Colab, use this to download the model files:

In [ ]:
# Uncomment these lines if running in Google Colab:
# from google.colab import files
# files.download('models/model.pt')
# files.download('models/labels.json')
# files.download('models/config.json')